## Scenario 1: A single data scientist participating in an ML competition

This scenario demonstrates how an individual data scientist can use MLflow to track machine learning experiments on their local machine. This is a common setup for solo projects, hackathons, or competitions where collaboration and remote access are not required.

### MLflow setup overview
- **Tracking server:** Not used (runs locally, no remote server)
- **Backend store:** Local filesystem (stores experiment metadata in the `mlruns/` folder)
- **Artifacts store:** Local filesystem (stores model files and other artifacts in the same `mlruns/` folder)

With this setup, all experiment runs, parameters, metrics, and artifacts are saved locally. You can explore and compare your experiments using the MLflow UI.

### How to use the MLflow UI
- You can launch the MLflow UI by running `mlflow ui` in your terminal, or by running the provided code cell in this notebook.
- The UI will be available at [http://localhost:5000](http://localhost:5000).
- Use the UI to browse experiments, compare runs, and inspect logged models and artifacts.

> **Tip:** This local setup is ideal for learning and prototyping. For team projects or production, you would typically use a remote tracking server and a more robust backend (e.g., a database and cloud storage).

In [6]:
%pip install mlflow
import mlflow

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [7]:
print(f"tracking URI: '{mlflow.get_tracking_uri()}'")

tracking URI: 'sqlite:////Users/salahmneimne/Desktop/MBD2025/TERM%202/MLOPS%20/IE-MLOPS-NYC-TAXIS/03-experiment-tracking/mlflow.db'


In [8]:
mlflow.search_experiments()

[<Experiment: artifact_location=('/Users/salahmneimne/Desktop/MBD2025/TERM 2/MLOPS '
  '/IE-MLOPS-NYC-TAXIS/03-experiment-tracking/mlruns/0'), creation_time=1772554473502, experiment_id='0', last_update_time=1772554473502, lifecycle_stage='active', name='Default', tags={}, workspace='default'>]

### Creating an experiment and logging a new run

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score, confusion_matrix
import numpy as np
import mlflow
import sklearn
import datetime
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

mlflow.set_experiment("my-experiment-1")

X, y = load_iris(return_X_y=True)
class_names = load_iris().target_names

models = [
    ("LogisticRegression", LogisticRegression(C=1.0, random_state=42, max_iter=1000, solver="lbfgs")),
    ("RandomForestClassifier", RandomForestClassifier(n_estimators=100, random_state=42))
]

for model_name, model in models:
    with mlflow.start_run() as run:
        mlflow.set_tag("model_type", model_name)
        mlflow.log_param("sklearn_version", sklearn.__version__)
        model.fit(X, y)
        y_pred = model.predict(X)
        acc = accuracy_score(y, y_pred)
        mlflow.log_metric("accuracy", acc)
        # Log confusion matrix as labeled DataFrame
        cm = confusion_matrix(y, y_pred)
        cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
        cm_df.to_csv("confusion_matrix_labeled.csv")
        mlflow.log_artifact("confusion_matrix_labeled.csv")
        # Confusion matrix heatmap as image
        plt.figure(figsize=(5,4))
        sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
        plt.title(f"Confusion Matrix ({model_name})")
        plt.ylabel("True label")
        plt.xlabel("Predicted label")
        plt.tight_layout()
        plt.savefig("confusion_matrix_heatmap.png")
        plt.close()
        mlflow.log_artifact("confusion_matrix_heatmap.png")
        # Provide input_example and use 'name' instead of deprecated 'artifact_path'
        input_example = np.expand_dims(X[0], axis=0)
        mlflow.sklearn.log_model(model, name="models", input_example=input_example)
        mlflow.set_tag("n_classes", len(np.unique(y)))
        mlflow.set_tag("run_time", datetime.datetime.now().isoformat())
        mlflow.set_tag("description", f"{model_name} on Iris dataset")
        print(f"Logged run for {model_name}, accuracy={acc:.3f}")

2026/03/03 17:26:13 INFO mlflow.tracking.fluent: Experiment with name 'my-experiment-1' does not exist. Creating a new experiment.
2026/03/03 17:26:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/03 17:26:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/03 17:26:16 WARNING mlflow.sklearn: Saving scikit-learn mod

2026/03/03 17:26:13 INFO mlflow.tracking.fluent: Experiment with name 'my-experiment-1' does not exist. Creating a new experiment.
2026/03/03 17:26:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/03 17:26:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/03 17:26:16 WARNING mlflow.sklearn: Saving scikit-learn mod

Logged run for LogisticRegression, accuracy=0.973
Logged run for RandomForestClassifier, accuracy=1.000
Logged run for RandomForestClassifier, accuracy=1.000


In [10]:
mlflow.search_experiments()

[<Experiment: artifact_location=('/Users/salahmneimne/Desktop/MBD2025/TERM 2/MLOPS '
  '/IE-MLOPS-NYC-TAXIS/03-experiment-tracking/mlruns/1'), creation_time=1772555173317, experiment_id='1', last_update_time=1772555173317, lifecycle_stage='active', name='my-experiment-1', tags={}, workspace='default'>,
 <Experiment: artifact_location=('/Users/salahmneimne/Desktop/MBD2025/TERM 2/MLOPS '
  '/IE-MLOPS-NYC-TAXIS/03-experiment-tracking/mlruns/0'), creation_time=1772554473502, experiment_id='0', last_update_time=1772554473502, lifecycle_stage='active', name='Default', tags={}, workspace='default'>]

In [11]:
# Launch MLflow UI (default: uses local mlruns/ folder)
import subprocess
import sys

# This will run 'mlflow ui' as a background process
subprocess.Popen([sys.executable, '-m', 'mlflow', 'ui'])
print("MLflow UI started. Open http://localhost:5000 in your browser.")

MLflow UI started. Open http://localhost:5000 in your browser.


MLflow UI started. Open http://localhost:5000 in your browser.


Backend store URI not provided. Using sqlite:///mlflow.db
Registry store URI not provided. Using backend store URI.


MLflow UI started. Open http://localhost:5000 in your browser.


Backend store URI not provided. Using sqlite:///mlflow.db
Registry store URI not provided. Using backend store URI.


2026/03/03 17:30:52 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/03 17:30:52 INFO mlflow.store.db.utils: Updating database tables
[MLflow] Security middleware enabled with default settings (localhost-only). To allow connections from other hosts, use --host 0.0.0.0 and configure --allowed-hosts and --cors-allowed-origins.
INFO:     Uvicorn running on http://127.0.0.1:5000 (Press CTRL+C to quit)
INFO:     Started parent process [85020]
[MLflow] Security middleware enabled with default settings (localhost-only). To allow connections from other hosts, use --host 0.0.0.0 and configure --allowed-hosts and --cors-allowed-origins.
INFO:     Uvicorn running on http://127.0.0.1:5000 (Press CTRL+C to quit)
INFO:     Started parent process [85020]
2026/03/03 17:30:54 INFO mlflow.server.jobs.utils: Starting huey consumer for job function optimize_prompts
2026/03/03 17:30:54 INFO mlflow.server.jobs.utils: Starting huey consumer for job function invoke_scorer
2026/03

MLflow UI started. Open http://localhost:5000 in your browser.


Backend store URI not provided. Using sqlite:///mlflow.db
Registry store URI not provided. Using backend store URI.


2026/03/03 17:30:52 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/03 17:30:52 INFO mlflow.store.db.utils: Updating database tables
[MLflow] Security middleware enabled with default settings (localhost-only). To allow connections from other hosts, use --host 0.0.0.0 and configure --allowed-hosts and --cors-allowed-origins.
INFO:     Uvicorn running on http://127.0.0.1:5000 (Press CTRL+C to quit)
INFO:     Started parent process [85020]
[MLflow] Security middleware enabled with default settings (localhost-only). To allow connections from other hosts, use --host 0.0.0.0 and configure --allowed-hosts and --cors-allowed-origins.
INFO:     Uvicorn running on http://127.0.0.1:5000 (Press CTRL+C to quit)
INFO:     Started parent process [85020]
2026/03/03 17:30:54 INFO mlflow.server.jobs.utils: Starting huey consumer for job function optimize_prompts
2026/03/03 17:30:54 INFO mlflow.server.jobs.utils: Starting huey consumer for job function invoke_scorer
2026/03

INFO:     127.0.0.1:57196 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:57196 - "GET /static-files/static/js/main.49592295.js HTTP/1.1" 200 OK
INFO:     127.0.0.1:57197 - "GET /static-files/static/css/main.e1790ccd.css HTTP/1.1" 200 OK
INFO:     127.0.0.1:57196 - "GET /static-files/TelemetryLogger.telemetry-worker.e586432cbf500042667b.worker.js HTTP/1.1" 200 OK
INFO:     127.0.0.1:57199 - "GET /static-files/favicon.ico HTTP/1.1" 200 OK
INFO:     127.0.0.1:57200 - "GET /static-files/manifest.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:57196 - "GET /ajax-api/3.0/mlflow/server-info HTTP/1.1" 200 OK
INFO:     127.0.0.1:57197 - "GET /ajax-api/3.0/mlflow/ui-telemetry HTTP/1.1" 200 OK
INFO:     127.0.0.1:57199 - "GET /static-files/favicon.ico HTTP/1.1" 200 OK
INFO:     127.0.0.1:57200 - "GET /static-files/manifest.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:57196 - "GET /ajax-api/3.0/mlflow/server-info HTTP/1.1" 200 OK
INFO:     127.0.0.1:57197 - "GET /ajax-api/3.0/mlflow/ui-telemetry HTTP/1.1" 2

### Interacting with the model registry

In [12]:
from mlflow.tracking import MlflowClient


client = MlflowClient()

In [14]:
from mlflow.exceptions import MlflowException

try:
    client.search_registered_models()
except MlflowException:
    print("It's not possible to access the model registry :(")

2026/03/03 17:31:55 INFO huey.consumer.Scheduler: Enqueueing periodic task mlflow.server.jobs.utils.online_scoring_scheduler: 02319bf5-4703-49ff-82db-b646302cea14.
[2026-03-03 17:31:55,942] INFO:huey.consumer.Scheduler:Scheduler:Enqueueing periodic task mlflow.server.jobs.utils.online_scoring_scheduler: 02319bf5-4703-49ff-82db-b646302cea14.


2026/03/03 17:31:55 INFO huey.consumer.Scheduler: Enqueueing periodic task mlflow.server.jobs.utils.online_scoring_scheduler: 02319bf5-4703-49ff-82db-b646302cea14.
[2026-03-03 17:31:55,942] INFO:huey.consumer.Scheduler:Scheduler:Enqueueing periodic task mlflow.server.jobs.utils.online_scoring_scheduler: 02319bf5-4703-49ff-82db-b646302cea14.


INFO:     127.0.0.1:57298 - "GET /ajax-api/2.0/mlflow/experiments/search?max_results=5&order_by=last_update_time+DESC HTTP/1.1" 200 OK


2026/03/03 17:31:55 INFO huey.consumer.Scheduler: Enqueueing periodic task mlflow.server.jobs.utils.online_scoring_scheduler: 02319bf5-4703-49ff-82db-b646302cea14.
[2026-03-03 17:31:55,942] INFO:huey.consumer.Scheduler:Scheduler:Enqueueing periodic task mlflow.server.jobs.utils.online_scoring_scheduler: 02319bf5-4703-49ff-82db-b646302cea14.


INFO:     127.0.0.1:57298 - "GET /ajax-api/2.0/mlflow/experiments/search?max_results=5&order_by=last_update_time+DESC HTTP/1.1" 200 OK


...

2026/03/03 17:33:55 INFO huey.consumer.Scheduler: Enqueueing periodic task mlflow.server.jobs.utils.online_scoring_scheduler: a9f8c480-daf6-4183-ac1d-9edb8600b013.
[2026-03-03 17:33:55,943] INFO:huey.consumer.Scheduler:Scheduler:Enqueueing periodic task mlflow.server.jobs.utils.online_scoring_scheduler: a9f8c480-daf6-4183-ac1d-9edb8600b013.
2026/03/03 17:33:58 INFO huey: Executing mlflow.server.jobs.utils.online_scoring_scheduler: a9f8c480-daf6-4183-ac1d-9edb8600b013
[2026-03-03 17:33:58,222] INFO:huey:Worker-1:Executing mlflow.server.jobs.utils.online_scoring_scheduler: a9f8c480-daf6-4183-ac1d-9edb8600b013
2026/03/03 17:33:58 INFO huey: mlflow.server.jobs.utils.online_scoring_scheduler: a9f8c480-daf6-4183-ac1d-9edb8600b013 executed in 0.012s
[2026-03-03 17:33:58,235] INFO:huey:Worker-1:mlflow.server.jobs.utils.online_scoring_scheduler: a9f8c480-daf6-4183-ac1d-9edb8600b013 executed in 0.012s
2026/03/03 17:33:58 INFO huey: Executing mlflow.server.jobs.utils.online_scoring_scheduler: a9f

INFO:     127.0.0.1:57536 - "GET /ajax-api/2.0/mlflow/experiments/search?max_results=25&order_by=last_update_time+DESC HTTP/1.1" 200 OK
INFO:     127.0.0.1:57558 - "POST /ajax-api/3.0/mlflow/ui-telemetry HTTP/1.1" 200 OK
INFO:     127.0.0.1:57558 - "POST /ajax-api/3.0/mlflow/ui-telemetry HTTP/1.1" 200 OK
INFO:     127.0.0.1:57598 - "GET /ajax-api/2.0/mlflow/experiments/search?max_results=25&order_by=last_update_time+DESC HTTP/1.1" 200 OK
INFO:     127.0.0.1:57598 - "GET /ajax-api/2.0/mlflow/experiments/search?max_results=25&order_by=last_update_time+DESC HTTP/1.1" 200 OK
INFO:     127.0.0.1:57603 - "GET /ajax-api/2.0/mlflow/experiments/search?max_results=25&order_by=last_update_time+DESC HTTP/1.1" 200 OK
INFO:     127.0.0.1:57603 - "GET /ajax-api/2.0/mlflow/experiments/search?max_results=25&order_by=last_update_time+DESC HTTP/1.1" 200 OK


2026/03/03 17:34:55 INFO huey.consumer.Scheduler: Enqueueing periodic task mlflow.server.jobs.utils.online_scoring_scheduler: 6cd451bb-1d24-46c5-a9eb-b8ad6be91b39.
[2026-03-03 17:34:55,926] INFO:huey.consumer.Scheduler:Scheduler:Enqueueing periodic task mlflow.server.jobs.utils.online_scoring_scheduler: 6cd451bb-1d24-46c5-a9eb-b8ad6be91b39.
2026/03/03 17:34:56 INFO huey: Executing mlflow.server.jobs.utils.online_scoring_scheduler: 6cd451bb-1d24-46c5-a9eb-b8ad6be91b39
[2026-03-03 17:34:56,063] INFO:huey:Worker-1:Executing mlflow.server.jobs.utils.online_scoring_scheduler: 6cd451bb-1d24-46c5-a9eb-b8ad6be91b39
2026/03/03 17:34:56 INFO huey: mlflow.server.jobs.utils.online_scoring_scheduler: 6cd451bb-1d24-46c5-a9eb-b8ad6be91b39 executed in 0.005s
[2026-03-03 17:34:56,068] INFO:huey:Worker-1:mlflow.server.jobs.utils.online_scoring_scheduler: 6cd451bb-1d24-46c5-a9eb-b8ad6be91b39 executed in 0.005s


INFO:     127.0.0.1:57671 - "GET /ajax-api/2.0/mlflow/experiments/search?max_results=25&order_by=last_update_time+DESC HTTP/1.1" 200 OK


2026/03/03 17:35:55 INFO huey.consumer.Scheduler: Enqueueing periodic task mlflow.server.jobs.utils.online_scoring_scheduler: 674369e6-0ccb-499f-abe1-968ff8ef026b.
[2026-03-03 17:35:55,925] INFO:huey.consumer.Scheduler:Scheduler:Enqueueing periodic task mlflow.server.jobs.utils.online_scoring_scheduler: 674369e6-0ccb-499f-abe1-968ff8ef026b.
2026/03/03 17:36:02 INFO huey: Executing mlflow.server.jobs.utils.online_scoring_scheduler: 674369e6-0ccb-499f-abe1-968ff8ef026b
[2026-03-03 17:36:02,666] INFO:huey:Worker-1:Executing mlflow.server.jobs.utils.online_scoring_scheduler: 674369e6-0ccb-499f-abe1-968ff8ef026b
2026/03/03 17:36:02 INFO huey: mlflow.server.jobs.utils.online_scoring_scheduler: 674369e6-0ccb-499f-abe1-968ff8ef026b executed in 0.015s
[2026-03-03 17:36:02,680] INFO:huey:Worker-1:mlflow.server.jobs.utils.online_scoring_scheduler: 674369e6-0ccb-499f-abe1-968ff8ef026b executed in 0.015s
2026/03/03 17:36:02 INFO huey: Executing mlflow.server.jobs.utils.online_scoring_scheduler: 674